In [44]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output

# file_path = "../rapidityTree/tree_Au197pnHFB14Au197pnHFB14_62p4_alpha3.00_cent70_80.root"
# file_path = "../rapidityTree/tree_Au197pnHFB14Au197pnHFB14_62p4_alpha3.00_cent70_80.root"

# file_path = "../rapidityTree/tree_Au197pnHFB14Au197pnHFB14_200_alpha3.00_cent0_5.root"
file_path = "../rapidityTree/tree_Au197pnHFB14Au197pnHFB14_62p4_alpha3.00_cent0_5.root"
# file_path = "../rapidityTree/tree_PbpnrwPbpnrw_17p3_alpha3.00_cent0_5.root"

with uproot.open(file_path) as f:
    tree = f["t"]
    arrays = tree.arrays(
        ["evt", "isProton", "y_final"],
        library="np"
    )

evt = arrays["evt"]
isProton = arrays["isProton"]
y_final = arrays["y_final"]


def proton_multiplicity(y_cut):
    # 选择 proton 且 |y| < y_cut
    # mask = (isProton == 1) & (np.abs(y_final) < y_cut)
    mask = (np.abs(y_final) < y_cut)

    evt_selected = evt[mask]

    # event-by-event 计数
    unique_evt, counts = np.unique(evt_selected, return_counts=True)

    return counts


In [46]:

def update(y_cut):
    clear_output(wait=True)

    mult = proton_multiplicity(y_cut)
    N = mult.astype(float)

    # cumulants
    c1 = N.mean()
    c2 = np.mean((N - c1)**2)
    c3 = np.mean((N - c1)**3)
    c4 = np.mean((N - c1)**4) - 3 * c2**2

    fig, ax = plt.subplots(figsize=(7, 4))

    # multiplicity histogram
    bins = np.arange(N.max() + 2) - 0.5
    ax.hist(N, bins=bins)

    ax.set_xlabel("Proton multiplicity")
    ax.set_ylabel("Events")
    ax.set_title(f"|y| < {y_cut:.2f}")
    ax.grid(True)

    # text box on the right
    text = (
        f"N = {len(N)}\n"
        f"c1 = {c1:.4f}\n"
        f"c2 = {c2:.4f}\n"
        f"c3 = {c3:.4f}\n"
        f"c4 = {c4:.4f}"
    )

    ax.text(
        1.02, 0.95,
        text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

    plt.tight_layout()
    plt.show()


slider = widgets.FloatSlider(
    value=0.5,
    min=0.01,
    max=7.0,
    step=0.1,
    description="y_cut",
    continuous_update=True
)


widgets.interact(update, y_cut=slider)

interactive(children=(FloatSlider(value=0.5, description='y_cut', max=7.0, min=0.01), Output()), _dom_classes=…

<function __main__.update(y_cut)>